# Pareja C — Publicar un release

Cuando un experimento (o un ensamble) mejora: **el F1 macro en validación no baja y el promedio de
los 4 débiles sube** contra el último release. `release.py` mide en val **y en test** (la única vez
que se mira test), copia el `.pt`, las métricas y la receta a `releases/<tag>/` y escribe `LEEME.md`.

Versiones: PARCHE `v0.2.1` (arreglo sin cambiar el modelo), MENOR `v0.3.0` (modelo nuevo que
mejora), MAYOR `v1.0.0` (la que se entrega al repo principal). Un release no se reescribe.

Flujo: rama `release-<tag>` → Pull Request → al fusionarlo, la etiqueta (celda final) → GitHub
Actions publica el release. El repo principal lo usa fijando `RELEASE_CLASIFICADOR` en un PR.

## Preparar el entorno

**CPU basta**: entrenar una cabeza con vectores ya calculados tarda segundos.

- `P2M_DATOS`: dataset compartido (solo lectura). `P2M_SALIDAS`: carpeta **compartida del equipo**
  (`poster2movie_clasificador`), donde quedan vectores, modelos y predicciones para las otras parejas.
  Agreguen las dos a *Mi unidad* como acceso directo.
- Trabaja en la rama **`main`** (se crea desde `main` si no existe).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["P2M_DATOS"] = "/content/drive/MyDrive/poster2movie_datos"            # solo lectura
os.environ["P2M_SALIDAS"] = "/content/drive/MyDrive/poster2movie_clasificador"   # compartida del equipo
REPO = "11909/Poster2Movie-Clasificador"
RAMA = "main"
DESTINO = "/content/clasificador"
if not os.path.isdir(DESTINO):
    !git clone -q --recurse-submodules https://github.com/{REPO}.git {DESTINO}
%cd {DESTINO}
!git fetch -q origin
!git checkout -q {RAMA} 2>/dev/null || git checkout -q -b {RAMA} origin/main
!git pull -q origin {RAMA} 2>/dev/null; git submodule update -q --init
!pip install -q -r requirements.txt

In [ ]:
TAG = "v0.2.0"                 # el siguiente número
EXPERIMENTO = "B4_focal_g2"    # el que se publica
NOTAS = "Qué cambió y por qué ayudó (va en el LEEME del release)"
RAMA = f"release-{TAG}"
!git checkout -q -b {RAMA}
!python -m clasificador.comparar v0.1.0 {EXPERIMENTO}   # contra el último release, en validación

In [ ]:
!python -m clasificador.release {TAG} {EXPERIMENTO} --notas "{NOTAS}"

## Subir a la rama `release-<tag>`

En Colab el repo se borra al cerrar: suban `resultados/` antes. Se sube a **`release-<tag>`**, nunca a
`main`; luego abran el Pull Request (lo revisa alguien de **otra** pareja). Necesitan un token de
GitHub con permiso de escritura (*Settings → Developer settings → Personal access tokens → Tokens (classic)*, con el permiso `repo` (los *fine-grained* no sirven: el repo es de otra cuenta personal)).

In [ ]:
from getpass import getpass
!git config user.name "{input('Tu nombre: ')}"
!git config user.email "{input('Tu correo de GitHub: ')}"
!git add releases resultados
!git commit -q -m "{RAMA}: {input('Qué hiciste (mensaje del commit): ')}"
token = getpass("Token de GitHub: ")
!git push -q https://{token}@github.com/{REPO}.git HEAD:{RAMA}
del token
print(f"Abre el Pull Request: https://github.com/{REPO}/compare/main...{RAMA}")

## Después de fusionar el Pull Request: la etiqueta

Al subir la etiqueta, GitHub Actions crea el release con los archivos de `releases/<tag>/`.

In [ ]:
from getpass import getpass
!git fetch -q origin && git checkout -q main && git pull -q origin main
!test -f releases/{TAG}/LEEME.md && git tag {TAG}
token = getpass("Token de GitHub: ")
!git push -q https://{token}@github.com/{REPO}.git {TAG}
del token
print(f"https://github.com/{REPO}/releases/tag/{TAG}")

## Probar con un cartel

In [ ]:
from google.colab import files
from PIL import Image
from clasificador.predictor import Clasificador
from p2m.config import GENEROS_ES

clasificador = Clasificador()  # el release más reciente
for nombre in files.upload():
    resultado = clasificador.predecir(Image.open(nombre).convert("RGB"))
    print(nombre, "->", ", ".join(GENEROS_ES[g] for g in resultado["generos"]))
    top = sorted(resultado["probabilidades"].items(), key=lambda x: -x[1])[:5]
    print("   ", ", ".join(f"{GENEROS_ES[g]} {p:.2f}" for g, p in top))